# DOMCS-EEG Colab Follow-up Security Evaluation

This notebook is designed for Google Colab and follows the Drive-based setup you specified.

It will:
- mount Google Drive
- validate the uploaded repo and the EEGMMIDB `.npz`
- reuse an existing checkpoint or train one seed automatically
- run clean B2T evaluation
- run follow-up security experiments:
  - FGSM
  - PGD
  - explicit 50 Hz line-noise robustness
- save CSVs and plots back to Drive

Run the notebook top to bottom after your Drive repo upload is fully complete.

In [ ]:
# Colab setup and package install
from google.colab import drive
drive.mount('/content/drive')

!pip -q install numpy pandas scipy scikit-learn matplotlib torch

In [ ]:
import os
import sys
import json
import time
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO_ROOT = '/content/drive/MyDrive/github repo 12apr26/DOMCS_EEG_GITHUB_FINAL_20260406_2305/DOMCS-EEG'
NPZ_PATH = '/content/drive/MyDrive/EEG_Q1_BIOMETRIC_PROJECT/02_PREPROCESSING/outputs/EEGMMIDB_win2s_step1s_fs128.npz'
OUT_ROOT = '/content/drive/MyDrive/EEG_Q1_BIOMETRIC_PROJECT/03_ATTACK_RESULTS_DOMCS'

AUTO_TRAIN_IF_MISSING = True
RUN_SMOKE_TEST = True
MAX_SUBJECTS_SMOKE = 10
MAX_PROBES_SMOKE = 5000

FGSM_EPS_LIST = [0.002, 0.005, 0.01, 0.02]
PGD_EPS_LIST = [0.005, 0.01, 0.02]
PGD_ALPHA = 0.0025
PGD_STEPS = 7
LINE_NOISE_AMPLITUDES_50HZ = [0.01, 0.03, 0.05, 0.1]

TRAIN_RUNS = ['r01', 'r02']
TEST_RUNS = ['r03', 'r04', 'r05', 'r06', 'r07', 'r08', 'r09', 'r10', 'r11', 'r12', 'r13', 'r14']
K_PROTOTYPES = 3
SEED = 1
ATTACK_IMPOSTOR_LIMIT = 20
ATTACK_BATCH_SIZE = 64

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
for subdir in ['checkpoints', 'csv', 'figures', 'logs']:
    Path(OUT_ROOT, subdir).mkdir(parents=True, exist_ok=True)

print('REPO_ROOT =', REPO_ROOT)
print('NPZ_PATH   =', NPZ_PATH)
print('OUT_ROOT   =', OUT_ROOT)

In [ ]:
# Fail early if the repo upload or npz is incomplete
repo_root = Path(REPO_ROOT)
npz_path = Path(NPZ_PATH)

required_repo_entries = [
    repo_root / 'scripts',
    repo_root / 'domcs_eeg',
    repo_root / 'checkpoints',
    repo_root / 'scripts' / 'core_framework.py',
]

missing_repo = [str(p) for p in required_repo_entries if not p.exists()]
if missing_repo:
    raise FileNotFoundError(
        'Repo upload looks incomplete or the DOMCS-EEG subfolder path is wrong. Missing: ' + json.dumps(missing_repo, indent=2)
    )

if not npz_path.exists():
    raise FileNotFoundError(f'NPZ file not found: {npz_path}')

sys.path.insert(0, str(repo_root / 'scripts'))
sys.path.insert(0, str(repo_root))

print('Repo validation passed.')
print('NPZ exists:', npz_path)

In [ ]:
# Load repo framework and align it with Drive paths
import core_framework as cf

cf.CFG['ROOT'] = str(repo_root)
cf.CFG['NPZ_PATH'] = str(npz_path)
cf.CFG['EXP_ROOT'] = str(Path(OUT_ROOT) / 'training_runs')
cf.CFG['PAPER_OUT'] = str(Path(OUT_ROOT) / 'paper_outputs')
cf.CFG['TRAIN_RUNS'] = TRAIN_RUNS
cf.CFG['TEST_RUNS'] = TEST_RUNS
cf.CFG['KMEANS_K'] = K_PROTOTYPES
cf.CFG['SEEDS'] = [SEED]
cf.CFG['GPU_ID'] = '0'

DEVICE = cf.setup_device('0')
print('Device:', DEVICE)

In [ ]:
# Inspect the NPZ and validate required arrays
with np.load(npz_path, allow_pickle=True) as data:
    npz_keys = sorted(list(data.files))
    if 'X' not in data.files:
        raise KeyError("NPZ is missing required array 'X'.")
    if not any(k in data.files for k in ['y', 'Y']):
        raise KeyError("NPZ is missing required label array 'y' or 'Y'.")
    if not any(k in data.files for k in ['session', 'runs']):
        raise KeyError("NPZ is missing required run array 'session' or 'runs'.")

    X_preview = data['X']
    Y_preview = data['y'] if 'y' in data.files else data['Y']
    R_preview = data['session'] if 'session' in data.files else data['runs']

print('NPZ keys:', npz_keys)
print('X shape :', X_preview.shape)
print('Y shape :', Y_preview.shape)
print('Runs    :', len(R_preview), 'entries')
print('Subjects:', len(np.unique(Y_preview)))

In [ ]:
# Load dataset and build the B2T split
X, Y, runs = cf.load_dataset(str(npz_path))
split = cf.build_b2t_split(X, Y, runs, seed=SEED)

print('Train windows:', split['X_train_np'].shape)
print('Test windows :', split['X_test_np'].shape)
print('Unique train subjects:', len(np.unique(split['Y_train_np'])))
print('Unique test subjects :', len(np.unique(split['Y_test_np'])))

In [ ]:
# Optional smoke-test reduction for first pass on Colab
def apply_smoke_subset(split_dict, max_subjects=MAX_SUBJECTS_SMOKE, max_probes=MAX_PROBES_SMOKE):
    if not RUN_SMOKE_TEST:
        return split_dict

    subset = dict(split_dict)
    keep_subjects = np.unique(subset['Y_train_np'])[:max_subjects]

    train_mask = np.isin(subset['Y_train_np'], keep_subjects)
    test_mask = np.isin(subset['Y_test_np'], keep_subjects)

    subset['X_train_np'] = subset['X_train_np'][train_mask]
    subset['Y_train_np'] = subset['Y_train_np'][train_mask]
    subset['state_train'] = subset['state_train'][train_mask]

    subset['X_test_np'] = subset['X_test_np'][test_mask][:max_probes]
    subset['Y_test_np'] = subset['Y_test_np'][test_mask][:max_probes]
    subset['state_test'] = subset['state_test'][test_mask][:max_probes]
    subset['runs_test'] = subset['runs_test'][test_mask][:max_probes]

    tr_idx_sub, val_idx_sub = cf.train_test_split(
        np.arange(len(subset['X_train_np'])),
        test_size=cf.CFG['VAL_FRAC'],
        stratify=subset['Y_train_np'],
        random_state=SEED,
    )
    subset['tr_idx_sub'] = tr_idx_sub
    subset['val_idx_sub'] = val_idx_sub
    return subset

eval_split = apply_smoke_subset(split)
print('Using smoke subset:', RUN_SMOKE_TEST)
print('Eval train windows:', len(eval_split['X_train_np']))
print('Eval test windows :', len(eval_split['X_test_np']))

In [ ]:
# Checkpoint discovery and training helpers
from glob import glob

class Train60EPCompatibleModel(nn.Module):
    def __init__(self, Cin=64, emb_dim=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(Cin, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ELU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ELU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.id_head = nn.Sequential(nn.Linear(256, emb_dim), nn.LayerNorm(emb_dim))
        self.cs_head = nn.Sequential(nn.Linear(256, emb_dim), nn.LayerNorm(emb_dim))
        self.state_head = nn.Sequential(nn.Linear(emb_dim, 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, x):
        f = self.backbone(x).squeeze(-1)
        z_id = F.normalize(self.id_head(f), dim=1)
        z_cs = F.normalize(self.cs_head(f), dim=1)
        return z_id, z_cs

def get_z_id(output):
    if isinstance(output, tuple):
        return output[0]
    return output

def find_existing_checkpoint(repo_root_path, out_root_path):
    candidates = [
        Path(repo_root_path) / 'checkpoints' / 'seed_1' / 'model_best.pt',
        Path(repo_root_path) / 'checkpoints' / 'seed_1' / 'checkpoint_best.pt',
    ]
    candidates.extend(Path(p) for p in glob(str(Path(out_root_path) / 'training_runs' / '**' / 'checkpoint_best.pt'), recursive=True))
    candidates.extend(Path(p) for p in glob(str(Path(out_root_path) / 'checkpoints' / '**' / '*.pt'), recursive=True))
    for path in candidates:
        if path.exists():
            return path
    return None

def load_checkpoint_into_model(ckpt_path, split_dict):
    num_classes = int(len(np.unique(split_dict['Y_train_np'][split_dict['tr_idx_sub']])))
    ckpt = torch.load(ckpt_path, map_location=cf.DEVICE)
    state = ckpt['model_state'] if 'model_state' in ckpt else ckpt
    state_keys = list(state.keys())

    if any(k.startswith('backbone.') for k in state_keys) and any('cs_head' in k for k in state_keys):
        model = Train60EPCompatibleModel(emb_dim=cf.CFG['EMB_DIM']).to(cf.DEVICE)
        architecture = 'train_60ep_compatible'
    else:
        model = cf.DOMCSModel(emb_dim=cf.CFG['EMB_DIM']).to(cf.DEVICE)
        architecture = 'core_framework_domcs'

    arc = cf.ArcFaceLayer(cf.CFG['EMB_DIM'], num_classes).to(cf.DEVICE)

    if 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
        if 'arc_state' in ckpt:
            try:
                arc.load_state_dict(ckpt['arc_state'])
            except Exception:
                pass
    else:
        model.load_state_dict(ckpt)
    model.eval()
    print('Loaded architecture:', architecture)
    return model, arc

def train_one_seed_if_needed(split_dict, out_root_path):
    ckpt_path = find_existing_checkpoint(REPO_ROOT, out_root_path)
    if ckpt_path is not None:
        print('Found existing checkpoint:', ckpt_path)
        return ckpt_path
    if not AUTO_TRAIN_IF_MISSING:
        raise FileNotFoundError('No checkpoint found and AUTO_TRAIN_IF_MISSING=False')

    run_dir = Path(out_root_path) / 'training_runs' / 'auto_seed1'
    run_dir.mkdir(parents=True, exist_ok=True)
    print('No checkpoint found. Training one seed into:', run_dir)
    _, _, _ = cf.train_one_seed(SEED, split_dict, str(run_dir), config=cf.CFG, resume=False, verbose=True)
    ckpt_path = run_dir / f'seed_{SEED}' / 'checkpoint_best.pt'
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Training finished but checkpoint not found: {ckpt_path}')
    return ckpt_path

checkpoint_path = train_one_seed_if_needed(split, OUT_ROOT)
model, arc_layer = load_checkpoint_into_model(checkpoint_path, eval_split)
print('Active checkpoint:', checkpoint_path)

In [ ]:
# Clean evaluation helpers
def extract_embeddings(model, X_np, Y_np, S_np, batch_size=512):
    model.eval()
    ds = cf.EEGDataset(X_np, Y_np, S_np)
    loader = cf.DataLoader(ds, batch_size=batch_size, shuffle=False)
    embs = []
    with torch.no_grad():
        for xb, _, _ in loader:
            z_id = get_z_id(model(xb.to(cf.DEVICE)))
            embs.append(z_id.cpu().numpy())
    E = np.concatenate(embs, axis=0)
    norms = np.linalg.norm(E, axis=1, keepdims=True) + 1e-12
    return E / norms

def build_gallery(E_train, Y_train, k=K_PROTOTYPES):
    pvecs, powner = cf.build_prototypes(E_train, Y_train, K=k)
    return pvecs, powner

def all_scores_from_embeddings(E_test, Y_test, pvecs, powner):
    sim = E_test @ pvecs.T
    genuine_scores = []
    impostor_scores = []
    for i in range(len(E_test)):
        yt = Y_test[i]
        mg = powner == yt
        mi = powner != yt
        genuine_scores.append(float(sim[i, mg].max()))
        impostor_scores.extend(sim[i, mi].tolist())
    return np.array(genuine_scores), np.array(impostor_scores)

def compute_threshold_metrics(genuine_scores, impostor_scores, threshold=None):
    scores = np.concatenate([genuine_scores, impostor_scores])
    labels = np.concatenate([np.ones(len(genuine_scores)), np.zeros(len(impostor_scores))]).astype(np.int32)
    auc_val, eer_val = cf.compute_auc_eer(scores, labels)
    fpr, tpr, thrs = cf.roc_curve(labels, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    tau = float(thrs[idx]) if threshold is None else float(threshold)
    far = float((impostor_scores >= tau).mean())
    frr = float((genuine_scores < tau).mean())
    return {
        'auc': float(auc_val),
        'eer': float(eer_val),
        'threshold': tau,
        'far': far,
        'frr': frr,
        'n_genuine': int(len(genuine_scores)),
        'n_impostor': int(len(impostor_scores)),
    }

E_train_clean = extract_embeddings(model, eval_split['X_train_np'], eval_split['Y_train_np'], eval_split['state_train'])
E_test_clean = extract_embeddings(model, eval_split['X_test_np'], eval_split['Y_test_np'], eval_split['state_test'])
pvecs_clean, powner_clean = build_gallery(E_train_clean, eval_split['Y_train_np'])
genuine_clean, impostor_clean = all_scores_from_embeddings(E_test_clean, eval_split['Y_test_np'], pvecs_clean, powner_clean)
clean_metrics = compute_threshold_metrics(genuine_clean, impostor_clean)

clean_df = pd.DataFrame([{'condition': 'clean', **clean_metrics}])
clean_df.to_csv(Path(OUT_ROOT) / 'csv' / 'clean_baseline.csv', index=False)
print(clean_df)

In [ ]:
# Attack helpers
def make_subject_proto_map(pvecs, powner):
    proto_map = {}
    for sid in np.unique(powner):
        proto_map[int(sid)] = pvecs[powner == sid]
    return proto_map

PROTO_MAP = make_subject_proto_map(pvecs_clean, powner_clean)
TARGET_SUBJECTS = sorted(list(PROTO_MAP.keys()))[:ATTACK_IMPOSTOR_LIMIT]

def normalize_rows(x, dim=1, eps=1e-12):
    return x / (torch.norm(x, dim=dim, keepdim=True) + eps)

def score_against_subject(model, xb, proto_matrix):
    proto_t = torch.tensor(proto_matrix, dtype=torch.float32, device=cf.DEVICE)
    proto_t = normalize_rows(proto_t, dim=1)
    z_id = get_z_id(model(xb))
    z_id = normalize_rows(z_id, dim=1)
    return torch.max(z_id @ proto_t.T, dim=1).values

def attack_fgsm(model, x_np, proto_matrix, eps):
    x = torch.tensor(x_np, dtype=torch.float32, device=cf.DEVICE, requires_grad=True)
    scores = score_against_subject(model, x, proto_matrix)
    loss = -scores.mean()
    model.zero_grad(set_to_none=True)
    loss.backward()
    adv = x + eps * x.grad.sign()
    return adv.detach().cpu().numpy().astype(np.float32)

def attack_pgd(model, x_np, proto_matrix, eps, alpha, steps):
    x0 = torch.tensor(x_np, dtype=torch.float32, device=cf.DEVICE)
    x = x0.clone().detach()
    for _ in range(steps):
        x.requires_grad_(True)
        scores = score_against_subject(model, x, proto_matrix)
        loss = -scores.mean()
        model.zero_grad(set_to_none=True)
        loss.backward()
        x = x.detach() + alpha * x.grad.sign()
        delta = torch.clamp(x - x0, min=-eps, max=eps)
        x = (x0 + delta).detach()
    return x.detach().cpu().numpy().astype(np.float32)

def add_line_noise_50hz(X_np, amplitude, sfreq=128.0):
    X_out = X_np.astype(np.float32).copy()
    n_t = X_out.shape[-1]
    t = np.arange(n_t, dtype=np.float32) / sfreq
    base = np.sin(2 * np.pi * 50.0 * t)[None, :]
    for i in range(len(X_out)):
        sig_std = float(X_out[i].std() + 1e-8)
        X_out[i] += amplitude * sig_std * base
    return X_out

def evaluate_embeddings(E_test, Y_test, threshold, genuine_ref=None, impostor_ref=None):
    genuine_scores, impostor_scores = all_scores_from_embeddings(E_test, Y_test, pvecs_clean, powner_clean)
    metrics = compute_threshold_metrics(genuine_scores, impostor_scores, threshold=threshold)
    metrics['genuine_score_drop'] = float((genuine_ref.mean() - genuine_scores.mean()) if genuine_ref is not None else 0.0)
    metrics['impostor_score_rise'] = float((impostor_scores.mean() - impostor_ref.mean()) if impostor_ref is not None else 0.0)
    return metrics, genuine_scores, impostor_scores

In [ ]:
# FGSM and PGD impersonation evaluation
def run_targeted_attack_grid(split_dict, mode='fgsm', eps_list=None, alpha=PGD_ALPHA, steps=PGD_STEPS):
    eps_list = eps_list or ([0.01] if mode == 'fgsm' else [0.01])
    X_test = split_dict['X_test_np']
    Y_test = split_dict['Y_test_np']
    threshold = clean_metrics['threshold']
    records = []

    for eps in eps_list:
        before_scores = []
        after_scores = []
        successes = []

        for target_sid in TARGET_SUBJECTS:
            proto = PROTO_MAP[target_sid]
            impostor_ids = [sid for sid in np.unique(Y_test) if sid != target_sid][:ATTACK_IMPOSTOR_LIMIT]

            for imp_sid in impostor_ids:
                idx = np.where(Y_test == imp_sid)[0]
                if len(idx) == 0:
                    continue
                chosen = idx[:ATTACK_BATCH_SIZE]
                xb = X_test[chosen]

                E_before = extract_embeddings(model, xb, np.full(len(xb), imp_sid), np.zeros(len(xb), dtype=np.int64), batch_size=ATTACK_BATCH_SIZE)
                score_before = (E_before @ normalize_rows(torch.tensor(proto, dtype=torch.float32), dim=1).cpu().numpy().T).max(axis=1)

                if mode == 'fgsm':
                    x_adv = attack_fgsm(model, xb, proto, eps)
                else:
                    x_adv = attack_pgd(model, xb, proto, eps, alpha, steps)

                E_after = extract_embeddings(model, x_adv, np.full(len(x_adv), imp_sid), np.zeros(len(x_adv), dtype=np.int64), batch_size=ATTACK_BATCH_SIZE)
                score_after = (E_after @ normalize_rows(torch.tensor(proto, dtype=torch.float32), dim=1).cpu().numpy().T).max(axis=1)

                before_scores.extend(score_before.tolist())
                after_scores.extend(score_after.tolist())
                successes.extend((score_after >= threshold).astype(np.int32).tolist())

        records.append({
            'attack': mode.upper(),
            'epsilon': float(eps),
            'alpha': float(alpha if mode == 'pgd' else 0.0),
            'steps': int(steps if mode == 'pgd' else 1),
            'threshold': float(threshold),
            'attack_success_rate': float(np.mean(successes) if successes else 0.0),
            'mean_impostor_score_before': float(np.mean(before_scores) if before_scores else 0.0),
            'mean_impostor_score_after': float(np.mean(after_scores) if after_scores else 0.0),
            'impostor_score_rise': float((np.mean(after_scores) - np.mean(before_scores)) if before_scores else 0.0),
            'clean_far': float(clean_metrics['far']),
            'clean_frr': float(clean_metrics['frr']),
            'clean_eer': float(clean_metrics['eer']),
            'clean_auc': float(clean_metrics['auc']),
        })
    return pd.DataFrame(records)

df_fgsm = run_targeted_attack_grid(eval_split, mode='fgsm', eps_list=FGSM_EPS_LIST)
df_pgd = run_targeted_attack_grid(eval_split, mode='pgd', eps_list=PGD_EPS_LIST, alpha=PGD_ALPHA, steps=PGD_STEPS)

df_fgsm.to_csv(Path(OUT_ROOT) / 'csv' / 'fgsm_attack_results.csv', index=False)
df_pgd.to_csv(Path(OUT_ROOT) / 'csv' / 'pgd_attack_results.csv', index=False)
display(df_fgsm)
display(df_pgd)

In [ ]:
# 50 Hz line-noise robustness on the verification set
line_records = []
for amp in LINE_NOISE_AMPLITUDES_50HZ:
    X_noisy = add_line_noise_50hz(eval_split['X_test_np'], amplitude=amp, sfreq=128.0)
    E_noisy = extract_embeddings(model, X_noisy, eval_split['Y_test_np'], eval_split['state_test'])
    metrics, genuine_scores, impostor_scores = evaluate_embeddings(
        E_noisy,
        eval_split['Y_test_np'],
        threshold=clean_metrics['threshold'],
        genuine_ref=genuine_clean,
        impostor_ref=impostor_clean,
    )
    metrics.update({
        'attack': 'LINE_NOISE_50HZ',
        'amplitude': float(amp),
    })
    line_records.append(metrics)

df_line = pd.DataFrame(line_records)
df_line.to_csv(Path(OUT_ROOT) / 'csv' / 'line_noise_50hz_results.csv', index=False)
display(df_line)

In [ ]:
# Merge summary CSV
summary_rows = []
summary_rows.append({'attack': 'CLEAN', 'setting': 'baseline', **clean_metrics, 'attack_success_rate': np.nan, 'genuine_score_drop': 0.0, 'impostor_score_rise': 0.0})

for _, row in df_fgsm.iterrows():
    summary_rows.append({
        'attack': row['attack'],
        'setting': f"eps={row['epsilon']}",
        'auc': clean_metrics['auc'],
        'eer': clean_metrics['eer'],
        'threshold': row['threshold'],
        'far': clean_metrics['far'],
        'frr': clean_metrics['frr'],
        'attack_success_rate': row['attack_success_rate'],
        'genuine_score_drop': np.nan,
        'impostor_score_rise': row['impostor_score_rise'],
    })

for _, row in df_pgd.iterrows():
    summary_rows.append({
        'attack': row['attack'],
        'setting': f"eps={row['epsilon']},steps={int(row['steps'])}",
        'auc': clean_metrics['auc'],
        'eer': clean_metrics['eer'],
        'threshold': row['threshold'],
        'far': clean_metrics['far'],
        'frr': clean_metrics['frr'],
        'attack_success_rate': row['attack_success_rate'],
        'genuine_score_drop': np.nan,
        'impostor_score_rise': row['impostor_score_rise'],
    })

for _, row in df_line.iterrows():
    summary_rows.append({
        'attack': row['attack'],
        'setting': f"amp={row['amplitude']}",
        'auc': row['auc'],
        'eer': row['eer'],
        'threshold': row['threshold'],
        'far': row['far'],
        'frr': row['frr'],
        'attack_success_rate': np.nan,
        'genuine_score_drop': row['genuine_score_drop'],
        'impostor_score_rise': row['impostor_score_rise'],
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(Path(OUT_ROOT) / 'csv' / 'attack_summary_merged.csv', index=False)
display(df_summary)

In [ ]:
# Publication-oriented plots
plt.figure(figsize=(7, 5))
plt.plot(df_fgsm['epsilon'], df_fgsm['attack_success_rate'], marker='o', label='FGSM ASR')
plt.plot(df_pgd['epsilon'], df_pgd['attack_success_rate'], marker='s', label='PGD ASR')
plt.xlabel('Epsilon')
plt.ylabel('Attack Success Rate')
plt.title('FGSM / PGD Impostor Attack Success Rate')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUT_ROOT) / 'figures' / 'fgsm_pgd_asr.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(df_fgsm['epsilon'], df_fgsm['impostor_score_rise'], marker='o', label='FGSM score rise')
plt.plot(df_pgd['epsilon'], df_pgd['impostor_score_rise'], marker='s', label='PGD score rise')
plt.xlabel('Epsilon')
plt.ylabel('Mean Impostor Score Rise')
plt.title('Score Shift Under White-box Attack')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUT_ROOT) / 'figures' / 'fgsm_pgd_score_shift.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(df_line['amplitude'], df_line['eer'], marker='o', label='EER')
plt.plot(df_line['amplitude'], df_line['auc'], marker='s', label='AUC')
plt.xlabel('50 Hz noise amplitude')
plt.ylabel('Metric value')
plt.title('50 Hz Line-noise Robustness')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUT_ROOT) / 'figures' / 'line_noise_50hz_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Final run manifest for reproducibility
manifest = {
    'repo_root': REPO_ROOT,
    'npz_path': NPZ_PATH,
    'out_root': OUT_ROOT,
    'checkpoint_path': str(checkpoint_path),
    'run_smoke_test': RUN_SMOKE_TEST,
    'max_subjects_smoke': MAX_SUBJECTS_SMOKE,
    'max_probes_smoke': MAX_PROBES_SMOKE,
    'fgsm_eps_list': FGSM_EPS_LIST,
    'pgd_eps_list': PGD_EPS_LIST,
    'pgd_alpha': PGD_ALPHA,
    'pgd_steps': PGD_STEPS,
    'line_noise_amplitudes_50hz': LINE_NOISE_AMPLITUDES_50HZ,
    'clean_metrics': clean_metrics,
}
with open(Path(OUT_ROOT) / 'logs' / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Saved outputs to:', OUT_ROOT)
print('Notebook complete.')